# Raw data audit

Sampling rate, annotation counts and trial durations for all task runs → which subjects to exclude.

In [1]:
import re, warnings
from pathlib import Path
from collections import Counter, defaultdict
import mne

DATA_PATH = Path.home() / "mne_data"
TASK_RUNS = set(range(3, 15))                 # runs 3–14 are the task runs
pat = re.compile(r"S(\d+)R(\d+)")

records = []
for f in sorted(DATA_PATH.rglob("S*R*.edf")):
    m = pat.search(f.stem)
    subj, run = int(m.group(1)), int(m.group(2))
    if run not in TASK_RUNS:
        continue
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
    dur, desc = raw.annotations.duration, raw.annotations.description
    records.append({
        "subject": subj, "run": run,
        "sfreq": raw.info["sfreq"], "n_annot": len(desc),
        "move_dur": [round(float(d),1) for d,l in zip(dur,desc) if l in ("T1","T2")],
        "rest_dur": [round(float(d),1) for d,l in zip(dur,desc) if l == "T0"],
    })

print(f"Scanned {len(records)} task-run files (runs 3–14, all subjects).\n")

# standards defined by the data (modal value = normal)
std_rate  = Counter(r["sfreq"]   for r in records).most_common(1)[0][0]
std_count = Counter(r["n_annot"] for r in records).most_common(1)[0][0]
print(f"Modal sampling rate   : {std_rate} Hz")
print(f"Modal annotations/run : {std_count}\n")

# 1) sampling rate per subject
subj_rates = defaultdict(set)
for r in records:
    subj_rates[r["subject"]].add(r["sfreq"])
off_rate = {s: sorted(v)[0] for s, v in subj_rates.items() if v != {std_rate}}
print("=== SUBJECTS WITH NON-STANDARD SAMPLING RATE ===")
for s in sorted(off_rate):
    print(f"  S{s:03d}: {off_rate[s]} Hz")
print(f"  ({len(off_rate)} subjects)\n")

# 2) annotation-count anomalies, showing the specific run(s)
bad = [r for r in records if r["n_annot"] != std_count]
by_sub = defaultdict(list)
for r in sorted(bad, key=lambda r:(r["subject"], r["run"])):
    by_sub[r["subject"]].append((r["run"], r["n_annot"]))
print("=== TASK RUNS WITH NON-STANDARD ANNOTATION COUNT ===")
for s in sorted(by_sub):
    runs = by_sub[s]
    if len(runs) == len(TASK_RUNS):
        print(f"  S{s:03d}: ALL task runs = {runs[0][1]} annotations")
    else:
        print(f"  S{s:03d}: " + ", ".join(f"R{rn} = {n}" for rn, n in runs))
print(f"  ({len(by_sub)} subjects affected)\n")

# 3) segment-duration distributions across the population
def dist(field, title):
    d = defaultdict(set)
    for r in records:
        for v in r[field]:
            d[v].add(r["subject"])
    print(f"=== {title} (rounded 0.1 s) ===")
    for v in sorted(d):
        subs = sorted(d[v])
        print(f"  {v:>4} s : " + (f"subjects {subs}" if len(subs) <= 6 else f"{len(subs)} subjects"))
    print()

dist("move_dur", "MOVEMENT (T1/T2) SEGMENT DURATIONS")
dist("rest_dur", "REST (T0) SEGMENT DURATIONS")

Scanned 1308 task-run files (runs 3–14, all subjects).

Modal sampling rate   : 160.0 Hz
Modal annotations/run : 30

=== SUBJECTS WITH NON-STANDARD SAMPLING RATE ===
  S088: 128.0 Hz
  S092: 128.0 Hz
  S100: 128.0 Hz
  (3 subjects)

=== TASK RUNS WITH NON-STANDARD ANNOTATION COUNT ===
  S088: ALL task runs = 38 annotations
  S089: R3 = 44
  S092: ALL task runs = 38 annotations
  S100: ALL task runs = 24 annotations
  S104: R8 = 26
  S106: R5 = 9
  (6 subjects affected)

=== MOVEMENT (T1/T2) SEGMENT DURATIONS (rounded 0.1 s) ===
   3.0 s : subjects [104]
   4.0 s : 10 subjects
   4.1 s : 106 subjects
   4.2 s : subjects [89]
   5.1 s : subjects [88, 92, 100]

=== REST (T0) SEGMENT DURATIONS (rounded 0.1 s) ===
   1.4 s : subjects [88, 92]
   3.1 s : subjects [106]
   4.0 s : subjects [89]
   4.1 s : 84 subjects
   4.2 s : 34 subjects
   4.3 s : 8 subjects
   5.1 s : subjects [100]



In [2]:
# inspect single-run deviations
dev_runs = defaultdict(list)
for r in records:
    if r["n_annot"] != std_count:
        dev_runs[r["subject"]].append(r["run"])

single_run_subjects = {
    s: sorted(runs) for s, runs in dev_runs.items()
    if len(runs) < len(TASK_RUNS) and s not in off_rate
}

def show_run(subj, run):
    f = next(p for p in DATA_PATH.rglob(f"S{subj:03d}R{run:02d}.edf"))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
    on, du, de = raw.annotations.onset, raw.annotations.duration, raw.annotations.description
    c = Counter(de)
    print(f"\n--- S{subj:03d} R{run:02d} | {raw.info['sfreq']:g} Hz | "
          f"duration {raw.n_times/raw.info['sfreq']:.1f} s | {len(de)} annotations ---")
    print(f"    labels: " + ", ".join(f"{k}={c[k]}" for k in ('T0','T1','T2')))
    print(f"    {'idx':>3}  {'onset':>7}  {'dur':>5}  label")
    for i, (o, d, lab) in enumerate(zip(on, du, de)):
        print(f"    {i:>3}  {o:>7.2f}  {d:>5.2f}  {lab}")

for s in sorted(single_run_subjects):
    print(f"==== S{s:03d}: deviating run(s) {single_run_subjects[s]} ====")
    for run in single_run_subjects[s]:
        show_run(s, run)

==== S089: deviating run(s) [3] ====

--- S089 R03 | 160 Hz | duration 181.0 s | 44 annotations ---
    labels: T0=22, T1=11, T2=11
    idx    onset    dur  label
      0     0.00   4.10  T0
      1     4.10   4.15  T2
      2     8.25   4.05  T0
      3    12.30   4.15  T1
      4    16.45   4.05  T0
      5    20.50   4.15  T1
      6    24.65   4.05  T0
      7    28.70   4.15  T2
      8    32.85   4.05  T0
      9    36.90   4.15  T2
     10    41.05   4.05  T0
     11    45.10   4.15  T1
     12    49.25   4.05  T0
     13    53.30   4.15  T1
     14    57.45   4.05  T0
     15    61.50   4.15  T2
     16    65.65   4.05  T0
     17    69.70   4.15  T2
     18    73.85   4.05  T0
     19    77.90   4.15  T1
     20    82.05   4.05  T0
     21    86.10   4.15  T2
     22    90.25   4.05  T0
     23    94.30   4.15  T1
     24    98.45   4.05  T0
     25   102.50   4.15  T2
     26   106.70   4.05  T0
     27   110.70   4.15  T1
     28   114.80   4.05  T0
     29   118.90   4.15  

## Audit: subject and run inclusion

1308 task-run files scanned (runs 3–14, 109 subjects). Standard: 160 Hz, 30 annotations per run, 4.1 s trials.

| Subject | Finding | Decision |
|---|---|---|
| S088, S092, S100 | 128 Hz, different cue timing (5.1 s trials) | **Exclude** |
| S089 | R3 is a longer recording: 22 regular trials | Keep |
| S104 | R8 cut off mid-trial: 13 trials, last one too short for the 4 s epoch → 12 epochs | Keep |
| S106 | R5 aborted after 4 regular trials (Task 2 transfer training run) | Keep |

Exclusion is based on sampling rate only.